In [ ]:
import pandas as pd
import numpy as np

# Carga del dataset y filtrado de casos en Barcelona

In [ ]:
# Cargar el archivo CSV
url = 'https://raw.githubusercontent.com/brunopless/TFM-Equipo-4-BDDS/main/entrega-3/df_con_NAs.csv'
data_bcn = pd.read_csv(url)

<ipython-input-3-b92f5e291221>:3: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  data_bcn = pd.read_csv(url)


# Limpieza del dataset

In [ ]:
# Ver qué columnas tienen valores NA
na_counts = data_bcn.isnull().sum()
na_columns = na_counts[na_counts > 0]
print(na_columns)

floor                             2146
exterior                          2422
rooms                                9
bathrooms                            9
status                               5
hasLift                           2244
topNewDevelopment                  622
isParkingSpaceIncludedInPrice    14922
ParkingSpacePrice                17153
media_precios_airbnbs_500            4
dtype: int64


##propertyType

In [ ]:
# Recodificar propertyType (One-Hot Encoding)
one_hot_propertyType = pd.get_dummies(data_bcn['propertyType'], prefix='propertyType')
data_bcn = pd.concat([data_bcn, one_hot_propertyType], axis=1)

In [ ]:
data_bcn['propertyType_chalet'] = data_bcn['propertyType_chalet'].replace({True: 1, False: 0})
data_bcn['propertyType_duplex'] = data_bcn['propertyType_duplex'].replace({True: 1, False: 0})
data_bcn['propertyType_flat'] = data_bcn['propertyType_flat'].replace({True: 1, False: 0})
data_bcn['propertyType_penthouse'] = data_bcn['propertyType_penthouse'].replace({True: 1, False: 0})
data_bcn['propertyType_studio'] = data_bcn['propertyType_studio'].replace({True: 1, False: 0})

##status

In [ ]:
data_bcn.status.unique()

array(['good', nan, 'newdevelopment', 'renew'], dtype=object)

In [ ]:
# Remover filas con valores perdidos en 'status'
data_bcn = data_bcn.dropna(subset=['status'])

In [ ]:
# Aplicar One-Hot Encoding a la columna 'status'
one_hot_status = pd.get_dummies(data_bcn['status'], prefix='status')
data_bcn = pd.concat([data_bcn, one_hot_status], axis=1)

In [ ]:
data_bcn['status_good'] = data_bcn['status_good'].replace({True: 1, False: 0})
data_bcn['status_newdevelopment'] = data_bcn['status_newdevelopment'].replace({True: 1, False: 0})
data_bcn['status_renew'] = data_bcn['status_renew'].replace({True: 1, False: 0})

##media_precios_airbnb_500

In [ ]:
# Remover filas con valores perdidos en 'media_precios_airbnbs_500'
data_bcn = data_bcn.dropna(subset=['media_precios_airbnbs_500'])

##rooms

In [ ]:
# Remover filas con valores perdidos en 'rooms'
data_bcn = data_bcn.dropna(subset=['rooms'])

In [ ]:
# Revisar qué columnas tienen valores NA
na_counts = data_bcn.isnull().sum()
na_columns = na_counts[na_counts > 0]
print(na_columns)

floor                             2141
exterior                          2417
hasLift                           2242
topNewDevelopment                  621
isParkingSpaceIncludedInPrice    14909
ParkingSpacePrice                17137
dtype: int64


##topNewDevelopment

In [ ]:
data_bcn.topNewDevelopment.unique()

array([False, True, nan], dtype=object)

In [ ]:
data_bcn['topNewDevelopment'] = data_bcn['topNewDevelopment'].replace({True: 1, False: 0})

In [ ]:
data_bcn['topNewDevelopment'].fillna(0, inplace=True)

In [ ]:
data_bcn['topNewDevelopment'].unique()

array([0., 1.])

##isParkingSpaceIncludedInPrice

In [ ]:
data_bcn.isParkingSpaceIncludedInPrice.unique()

array([nan,  1.,  0.])

In [ ]:
# Contar las filas donde ambas columnas tienen NA
na_count = data_bcn[data_bcn['isParkingSpaceIncludedInPrice'].isna() & data_bcn['ParkingSpacePrice'].isna()].shape[0]

print(f'Número de filas donde ambas columnas tienen NA: {na_count}')

Número de filas donde ambas columnas tienen NA: 14909


In [ ]:
data_bcn['isParkingSpaceIncludedInPrice'].fillna(0, inplace=True)

In [ ]:
data_bcn.loc[data_bcn['isParkingSpaceIncludedInPrice'] == 0, 'ParkingSpacePrice'] = 0

In [ ]:
# Revisar qué columnas tienen valores NA
na_counts = data_bcn.isnull().sum()
na_columns = na_counts[na_counts > 0]
print(na_columns)

floor                2141
exterior             2417
hasLift              2242
ParkingSpacePrice    2227
dtype: int64


In [ ]:
data_bcn = data_bcn.drop(columns=['ParkingSpacePrice']) #los valores son 0 o NA

#hasLift

In [ ]:
# Obtener los tipos de datos de cada columna
tipos_de_datos = data_bcn.dtypes
pd.set_option('display.max_rows', None)
# Mostrar los tipos de datos
print(tipos_de_datos)

Unnamed: 0                            int64
index                                 int64
Fecha extraccion                      int64
floor                                object
price                               float64
propertyType                         object
operation                            object
size                                float64
exterior                            float64
rooms                               float64
bathrooms                           float64
address                              object
province                             object
district                             object
country                              object
neighborhood                         object
latitude                            float64
longitude                           float64
status                               object
newDevelopment                        int64
hasLift                              object
topNewDevelopment                   float64
isParkingSpaceIncludedInPrice   

In [ ]:
data_bcn.hasLift.unique()

array([False, True, nan, 'True', 'False', '0', '1'], dtype=object)

In [ ]:
data_bcn['hasLift'] = data_bcn['hasLift'].replace({True: 1, 'True': 1, '1': 1, False: 0, 'False': 0, '0': 0})

In [ ]:
data_bcn.hasLift.unique()

array([ 0.,  1., nan])

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# Crear una copia del dataframe para comparar
data_bcn_knn = data_bcn.copy()

# Seleccionar las columnas relevantes para KNN (eliminar las que no influyen en el cálculo)
cols_to_impute = ['size', 'rooms', 'bathrooms', 'hasLift', 'log_price_m2',
                  'propertyType_chalet', 'propertyType_duplex',
                  'propertyType_flat', 'propertyType_penthouse', 'propertyType_studio']

# Filtrar el dataframe solo con esas columnas
data_knn = data_bcn_knn[cols_to_impute]

# Aplicar el imputador KNN
imputer = KNNImputer(n_neighbors=5)
data_knn_imputed = pd.DataFrame(imputer.fit_transform(data_knn), columns=cols_to_impute)

# Reemplazar la columna 'hasLift' en el DataFrame original con los valores imputados
data_bcn_knn['hasLift'] = data_knn_imputed['hasLift'].round().astype(int)  # Redondear y convertir a entero


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Crear una copia del dataframe para comparar
data_bcn_rf = data_bcn.copy()

# Filtrar los registros con valores no nulos en 'hasLift'
data_rf_train = data_bcn_rf[data_bcn_rf['hasLift'].notna()]

# Filtrar las columnas relevantes para el modelo
features = ['size', 'rooms', 'bathrooms', 'log_price_m2',
            'propertyType_chalet', 'propertyType_duplex', 'propertyType_flat',
            'propertyType_penthouse', 'propertyType_studio']
X = data_rf_train[features]
y = data_rf_train['hasLift']

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar un modelo de Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predecir los valores faltantes en 'hasLift' para todo el dataset
data_rf_missing = data_bcn_rf[data_bcn_rf['hasLift'].isna()]
data_bcn_rf.loc[data_rf_missing.index, 'hasLift'] = rf_model.predict(data_rf_missing[features])

# Evaluar la precisión del modelo en los datos de prueba
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Precisión del modelo Random Forest: {accuracy}')


Precisión del modelo Random Forest: 0.7184466019417476


In [ ]:
# Comparación de resultados
comparison_df = pd.DataFrame({
    'Original': data_bcn['hasLift'],
    'KNN Imputed': data_bcn_knn['hasLift'],
    'RF Imputed': data_bcn_rf['hasLift']
})

# Mostrar las primeras filas de la comparación
print(comparison_df.head())

   Original  KNN Imputed  RF Imputed
0       0.0          0.0         0.0
1       1.0          1.0         1.0
2       NaN          1.0         1.0
3       1.0          1.0         1.0
4       1.0          1.0         1.0


In [ ]:
pd.set_option('display.max_rows', None)
# Mostrar los tipos de datos
print(comparison_df)

       Original  KNN Imputed  RF Imputed
0           0.0          0.0         0.0
1           1.0          1.0         1.0
2           NaN          1.0         1.0
3           1.0          1.0         1.0
4           1.0          1.0         1.0
5           1.0          1.0         1.0
6           1.0          1.0         1.0
7           1.0          1.0         1.0
8           1.0          1.0         1.0
9           1.0          1.0         1.0
10          1.0          1.0         1.0
11          1.0          1.0         1.0
12          1.0          1.0         1.0
13          1.0          1.0         1.0
14          NaN          1.0         1.0
15          1.0          1.0         1.0
16          1.0          1.0         1.0
17          NaN          0.0         1.0
18          1.0          1.0         1.0
19          1.0          1.0         1.0
20          1.0          1.0         1.0
21          1.0          1.0         1.0
22          1.0          1.0         1.0
23          1.0 

In [ ]:
# Contar los valores NaN en cada columna del DataFrame de comparación
nan_counts = comparison_df.isna().sum()

# Mostrar el resultado
print(nan_counts)

Original       2242
KNN Imputed      18
RF Imputed        0
dtype: int64


In [ ]:
# Reemplazar la columna 'hasLift' en data_bcn con los valores imputados por RF
data_bcn['hasLift'] = data_bcn_rf['hasLift']

In [ ]:
# Guardar el DataFrame en un archivo CSV
data_bcn.to_csv('df_bcn_imputado.csv', index=False)

# Descargar el archivo CSV
from google.colab import files
files.download('df_bcn_imputado.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>